# KOSPI 주가 방향 예측

> **Runtime → GPU 설정 권장**: Runtime > Change runtime type > T4 GPU

| Step | 설명 |
|------|------|
| 1 | OHLCV 데이터 수집 (pykrx) |
| 2 | 기술적 지표 피처 엔지니어링 |
| 3 | FinBERT 뉴스 감성 분석 |
| 4 | 시계열 데이터셋 생성 |
| 5 | ARIMA / LSTM / Transformer 학습 |
| 6 | 평가 및 시각화 |

각 Step은 중간 파일(csv/npz)을 저장하므로 **이미 파일이 있으면 해당 셀을 스킵**합니다.

In [ ]:
!pip install -q pykrx statsmodels transformers accelerate

In [ ]:
# Google Drive 마운트 (런타임 재시작 후에도 데이터 유지)
# Drive 사용 안 할 경우 False 로 변경
USE_DRIVE = True

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_DIR = '/content/drive/MyDrive/ml_project'
else:
    BASE_DIR = '/content/ml_project'

print(f'BASE_DIR: {BASE_DIR}')

In [ ]:
import os, time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import MinMaxScaler

# 경로
DATA_DIR   = os.path.join(BASE_DIR, 'data')
MODEL_DIR  = os.path.join(BASE_DIR, 'models')
RESULT_DIR = os.path.join(BASE_DIR, 'results')
for d in [DATA_DIR, MODEL_DIR, RESULT_DIR]:
    os.makedirs(d, exist_ok=True)

# 설정값
WINDOW_SIZE      = 20
TEST_RATIO       = 0.2
SEED             = 42
TARGET_THRESHOLD = 0.005
START_DATE       = '20060101'
END_DATE         = '20260401'
STOCK_CODES_PATH = os.path.join(BASE_DIR, 'kospi_stock_codes.csv')
NEWS_PATH        = os.path.join(BASE_DIR, 'kospi_news.csv')

np.random.seed(SEED)
torch.manual_seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

# 한글 폰트
import subprocess
subprocess.run(['apt-get', 'install', '-y', 'fonts-nanum'], capture_output=True)
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
fm.fontManager.addfont('/usr/share/fonts/truetype/nanum/NanumGothic.ttf')
plt.rcParams['font.family'] = 'NanumGothic'
plt.rcParams['axes.unicode_minus'] = False
print('설정 완료')

## 데이터 파일 업로드

`kospi_stock_codes.csv`, `kospi_news.csv` 파일이 필요합니다.  
Drive를 사용하는 경우 `ml_project/` 폴더에 미리 올려두면 자동으로 인식합니다.

In [ ]:
files_needed = {
    STOCK_CODES_PATH: 'kospi_stock_codes.csv',
    NEWS_PATH:        'kospi_news.csv',
}

missing = [name for path, name in files_needed.items() if not os.path.exists(path)]

if missing:
    from google.colab import files as colab_files
    print(f'다음 파일을 업로드하세요: {missing}')
    uploaded = colab_files.upload()
    for fname, data in uploaded.items():
        dest = os.path.join(BASE_DIR, fname)
        with open(dest, 'wb') as f:
            f.write(data)
        print(f'  업로드 완료: {dest}')
else:
    print('파일 확인 완료')

## Step 1: OHLCV 데이터 수집

`ohlcv.csv`가 이미 있으면 수집을 스킵합니다.

In [ ]:
OHLCV_PATH = os.path.join(DATA_DIR, 'ohlcv.csv')

def fetch_ohlcv(code, start, end):
    from pykrx import stock as krx
    df = krx.get_market_ohlcv_by_date(start, end, code)
    df = df.reset_index()
    df.rename(columns={
        '날짜': 'date', '시가': 'open', '고가': 'high',
        '저가': 'low', '종가': 'close', '거래량': 'volume'
    }, inplace=True)
    df['code'] = code
    return df

if os.path.exists(OHLCV_PATH):
    print('ohlcv.csv 이미 존재 → 로드')
    df_ohlcv = pd.read_csv(OHLCV_PATH, parse_dates=['date'])
else:
    df_codes = pd.read_csv(STOCK_CODES_PATH, dtype={'종목코드': str})
    codes = df_codes['종목코드'].str.zfill(6).tolist()
    all_data = []
    for i, code in enumerate(codes, 1):
        print(f'[{i}/{len(codes)}] {code}', end=' ')
        try:
            df = fetch_ohlcv(code, START_DATE, END_DATE)
            all_data.append(df)
            print(f'{len(df)}행')
        except Exception as e:
            print(f'실패: {e}')
        time.sleep(0.3)
    df_ohlcv = pd.concat(all_data, ignore_index=True)
    df_ohlcv.to_csv(OHLCV_PATH, index=False, encoding='utf-8-sig')
    print(f'저장: {OHLCV_PATH}')

n_codes = df_ohlcv['code'].nunique()
print(f'총 {len(df_ohlcv)}행, {n_codes}종목')

## Step 2: 피처 엔지니어링

기술적 지표 생성: SMA(5/20/60), RSI(14), MACD, Bollinger Bands, 로그수익률, 타겟 레이블

In [ ]:
FEATURES_PATH = os.path.join(DATA_DIR, 'features.csv')

def process_stock(df):
    df = df.sort_values('date').reset_index(drop=True)
    df['log_return'] = np.log(df['close'] / df['close'].shift(1))
    for w in [5, 20, 60]:
        df[f'sma_{w}'] = df['close'].rolling(w).mean()
    delta = df['close'].diff()
    gain  = delta.where(delta > 0, 0.0).rolling(14).mean()
    loss  = (-delta.where(delta < 0, 0.0)).rolling(14).mean()
    df['rsi'] = 100 - (100 / (1 + gain / loss))
    ema_fast = df['close'].ewm(span=12).mean()
    ema_slow = df['close'].ewm(span=26).mean()
    df['macd']        = ema_fast - ema_slow
    df['macd_signal'] = df['macd'].ewm(span=9).mean()
    df['macd_hist']   = df['macd'] - df['macd_signal']
    sma20 = df['close'].rolling(20).mean()
    std20 = df['close'].rolling(20).std()
    df['bb_upper'] = sma20 + 2 * std20
    df['bb_lower'] = sma20 - 2 * std20
    df['bb_width'] = df['bb_upper'] - df['bb_lower']
    df['future_return'] = df['close'].shift(-1) / df['close'] - 1
    df['target'] = np.nan
    df.loc[df['future_return'] >  TARGET_THRESHOLD, 'target'] = 1
    df.loc[df['future_return'] < -TARGET_THRESHOLD, 'target'] = 0
    return df.dropna().reset_index(drop=True)

if os.path.exists(FEATURES_PATH):
    print('features.csv 이미 존재 → 로드')
    df_features = pd.read_csv(FEATURES_PATH, parse_dates=['date'])
else:
    results = []
    for code, group in df_ohlcv.groupby('code'):
        results.append(process_stock(group.copy()))
    df_features = pd.concat(results, ignore_index=True)
    df_features.to_csv(FEATURES_PATH, index=False, encoding='utf-8-sig')
    print(f'저장: {FEATURES_PATH}')

print(f'총 {len(df_features)}행')

## Step 3: FinBERT 감성 분석

GPU 사용 시 약 30~60분 소요. `sentiment.csv`가 있으면 스킵합니다.

In [ ]:
SENTIMENT_PATH = os.path.join(DATA_DIR, 'sentiment.csv')

def load_finbert():
    from transformers import AutoTokenizer, AutoModelForSequenceClassification
    name = 'ProsusAI/finbert'
    tok = AutoTokenizer.from_pretrained(name)
    mdl = AutoModelForSequenceClassification.from_pretrained(name).to(DEVICE)
    mdl.eval()
    return tok, mdl

def predict_sentiment(texts, tok, mdl, batch_size=64):
    import torch.nn.functional as F
    out = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        inp = tok(batch, padding=True, truncation=True, max_length=512, return_tensors='pt')
        inp = {k: v.to(DEVICE) for k, v in inp.items()}
        with torch.no_grad():
            probs = F.softmax(mdl(**inp).logits, dim=-1).cpu()
        for p in probs:
            out.append(p[0].item() - p[1].item())  # positive - negative
    return out

def add_sentiment_time_features(df):
    df = df.copy()
    df['종목코드'] = df['종목코드'].astype(str).str.zfill(6)
    df['date'] = pd.to_datetime(df['date'])
    df = df.sort_values(['종목코드', 'date']).reset_index(drop=True)
    for col in ['sentiment_mean', 'sentiment_std', 'news_count']:
        df[col] = df[col].fillna(0)
    g = df.groupby('종목코드', group_keys=False)
    df['sentiment_lag1']   = g['sentiment_mean'].shift(1)
    df['sentiment_lag2']   = g['sentiment_mean'].shift(2)
    df['sentiment_change'] = g['sentiment_mean'].diff()
    roll_mean = g['news_count'].transform(lambda x: x.shift(1).rolling(20, min_periods=5).mean())
    roll_std  = g['news_count'].transform(lambda x: x.shift(1).rolling(20, min_periods=5).std())
    df['news_count_zscore_20'] = (df['news_count'] - roll_mean) / (roll_std + 1e-6)
    new_cols = ['sentiment_lag1', 'sentiment_lag2', 'sentiment_change', 'news_count_zscore_20']
    df[new_cols] = df[new_cols].replace([float('inf'), float('-inf')], 0).fillna(0)
    return df

if os.path.exists(SENTIMENT_PATH):
    print('sentiment.csv 이미 존재 → 로드')
    df_sentiment = pd.read_csv(SENTIMENT_PATH, parse_dates=['date'])
else:
    df_news = pd.read_csv(NEWS_PATH).dropna(subset=['제목'])
    print(f'뉴스 {len(df_news)}건 로드')
    tok, finbert = load_finbert()
    texts = df_news['제목'].tolist()
    print(f'감성 분석 중... ({len(texts)}건)')
    df_news['sentiment_score'] = predict_sentiment(texts, tok, finbert)
    df_news['date'] = pd.to_datetime(
        df_news['날짜'].str.strip().str[:10], format='%Y.%m.%d', errors='coerce'
    )
    df_news = df_news.dropna(subset=['date'])
    daily = df_news.groupby(['종목코드', 'date']).agg(
        sentiment_mean=('sentiment_score', 'mean'),
        sentiment_std =('sentiment_score', 'std'),
        news_count    =('sentiment_score', 'count'),
    ).reset_index()
    daily['sentiment_std'] = daily['sentiment_std'].fillna(0)
    df_sentiment = add_sentiment_time_features(daily)
    df_sentiment.to_csv(SENTIMENT_PATH, index=False, encoding='utf-8-sig')
    print(f'저장: {SENTIMENT_PATH}')

print(f'총 {len(df_sentiment)}행')

## Step 4: 시계열 데이터셋 생성

In [ ]:
DATASET_PATH = os.path.join(DATA_DIR, 'dataset.npz')

FEATURE_COLS = [
    'log_return', 'volume',
    'sma_5', 'sma_20', 'sma_60',
    'rsi', 'macd', 'macd_signal', 'macd_hist',
    'bb_upper', 'bb_lower', 'bb_width',
    'sentiment_mean_weighted', 'sentiment_std_weighted',
    'sentiment_lag1', 'sentiment_lag2', 'sentiment_change', 'news_count_zscore_20',
]

def create_sequences(df, window, cols):
    data   = df[cols].values
    target = df['target'].values
    X, y = [], []
    for i in range(window, len(data)):
        X.append(data[i-window:i])
        y.append(target[i])
    return np.array(X), np.array(y)

if os.path.exists(DATASET_PATH):
    print('dataset.npz 이미 존재 → 로드')
    _d = np.load(DATASET_PATH)
    X_train, X_test = _d['X_train'], _d['X_test']
    y_train, y_test = _d['y_train'], _d['y_test']
    log_returns_raw = _d['log_returns_raw'] if 'log_returns_raw' in _d.files else None
    if log_returns_raw is None:
        print('  ⚠ log_returns_raw 없음 — 아래 셀에서 features.csv로 복원합니다')
else:
    df_feat = df_features.copy()
    df_feat['code'] = df_feat['code'].astype(str).str.zfill(6)
    df_sent = df_sentiment.copy()
    df_sent.rename(columns={'종목코드': 'code'}, inplace=True)
    df_sent['code'] = df_sent['code'].astype(str).str.zfill(6)
    df = df_feat.merge(df_sent, on=['code', 'date'], how='left')
    sent_cols = ['sentiment_mean', 'sentiment_std', 'news_count',
                 'sentiment_lag1', 'sentiment_lag2', 'sentiment_change', 'news_count_zscore_20']
    for col in sent_cols:
        if col not in df.columns:
            df[col] = 0
    df[sent_cols] = df[sent_cols].fillna(0)
    weight = np.log1p(df['news_count'])
    df['sentiment_mean_weighted'] = df['sentiment_mean'] * weight
    df['sentiment_std_weighted']  = df['sentiment_std']  * weight
    df = df.dropna(subset=FEATURE_COLS + ['target'])

    all_Xtr, all_ytr, all_Xte, all_yte = [], [], [], []
    all_lr_raw = []
    used = skipped = 0

    for code, group in df.groupby('code'):
        group = group.sort_values('date').reset_index(drop=True)
        if len(group) < WINDOW_SIZE + 2:
            skipped += 1; continue
        split   = int(len(group) * (1 - TEST_RATIO))
        train_g = group.iloc[:split].copy()
        test_g  = group.iloc[split - WINDOW_SIZE:].copy()
        if len(train_g) < WINDOW_SIZE + 1 or len(test_g) < WINDOW_SIZE + 1:
            skipped += 1; continue

        # 정규화 전 원본 log_return 저장
        test_lr_raw = test_g['log_return'].values

        scaler = MinMaxScaler()
        train_g[FEATURE_COLS] = scaler.fit_transform(train_g[FEATURE_COLS])
        test_g[FEATURE_COLS]  = scaler.transform(test_g[FEATURE_COLS])
        Xtr, ytr = create_sequences(train_g, WINDOW_SIZE, FEATURE_COLS)
        Xte, yte = create_sequences(test_g,  WINDOW_SIZE, FEATURE_COLS)
        all_Xtr.append(Xtr); all_ytr.append(ytr)
        all_Xte.append(Xte); all_yte.append(yte)
        all_lr_raw.append(test_lr_raw[WINDOW_SIZE:])  # 타겟 시점과 정렬
        used += 1

    X_train = np.concatenate(all_Xtr); y_train = np.concatenate(all_ytr)
    X_test  = np.concatenate(all_Xte); y_test  = np.concatenate(all_yte)
    log_returns_raw = np.concatenate(all_lr_raw)
    np.savez(DATASET_PATH, X_train=X_train, X_test=X_test,
             y_train=y_train, y_test=y_test, log_returns_raw=log_returns_raw)
    print(f'저장: {DATASET_PATH}  (사용 {used}종목, 제외 {skipped}종목)')

print(f'Train: {X_train.shape}  Test: {X_test.shape}')
print(f'y_test  하락(0): {(y_test==0).sum()}  상승(1): {(y_test==1).sum()}')
if log_returns_raw is not None:
    print(f'log_returns_raw: {log_returns_raw.shape}')

In [ ]:
# 기존 dataset.npz에 log_returns_raw가 없을 경우 features.csv에서 복원
# dataset.npz를 새로 만들었다면 이 셀은 자동으로 스킵됩니다.
if log_returns_raw is None and os.path.exists(FEATURES_PATH):
    print('features.csv에서 log_returns_raw 복원 중...')
    df_f = pd.read_csv(FEATURES_PATH, parse_dates=['date'])
    df_f['code'] = df_f['code'].astype(str).str.zfill(6)
    all_lr_raw = []
    for code, group in df_f.groupby('code'):
        group = group.sort_values('date').reset_index(drop=True)
        if len(group) < WINDOW_SIZE + 2:
            continue
        split  = int(len(group) * (1 - TEST_RATIO))
        test_g = group.iloc[split - WINDOW_SIZE:]
        if len(test_g) < WINDOW_SIZE + 1:
            continue
        all_lr_raw.append(test_g['log_return'].values[WINDOW_SIZE:])
    log_returns_raw = np.concatenate(all_lr_raw)
    # 기존 npz에 추가 저장
    _d2 = np.load(DATASET_PATH)
    np.savez(DATASET_PATH, **{k: _d2[k] for k in _d2.files},
             log_returns_raw=log_returns_raw)
    print(f'복원 완료: {log_returns_raw.shape}')
elif log_returns_raw is not None:
    print(f'log_returns_raw 확인: {log_returns_raw.shape}')

# ── ARIMA (AR(5) baseline) ──────────────────────────────────────────────
def train_arima(X_train, X_test, y_train, y_test):
    from statsmodels.tsa.ar_model import AutoReg
    p            = 5
    lr_idx       = 0
    train_series = X_train[:, -1, lr_idx]
    neutral      = float(np.mean(train_series))
    print(f'  train_series: {len(train_series):,}개  neutral threshold: {neutral:.4f}')
    try:
        print('  AR(5) 계수 추정 중...')
        fitted    = AutoReg(train_series, lags=p, old_names=False).fit()
        intercept = fitted.params[0]
        ar_coefs  = fitted.params[1:]
        print(f'  intercept={intercept:.4f}  AR coefs={ar_coefs.round(4).tolist()}')
        windows   = X_test[:, -p:, lr_idx]
        forecasts = intercept + windows @ ar_coefs[::-1]
        preds     = (forecasts > neutral).astype(int)
        up_ratio  = preds.mean()
        print(f'  예측 완료 — 상승(1): {preds.sum():,} ({up_ratio:.1%})  '
              f'하락(0): {(preds==0).sum():,} ({1-up_ratio:.1%})')
    except Exception as e:
        print(f'  ARIMA 실패 (zeros로 대체): {e}')
        preds = np.zeros(len(y_test), dtype=int)
    return preds

# ── LSTM (hidden 128, dropout 0.3) ──────────────────────────────────────
class LSTMClassifier(nn.Module):
    def __init__(self, input_size, hidden=128, layers=2, dropout=0.3):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden, layers,
                            batch_first=True, dropout=dropout)
        self.fc   = nn.Linear(hidden, 1)
    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :]).squeeze(-1)

# ── Transformer (d_model 128, ff 256) ───────────────────────────────────
class TransformerClassifier(nn.Module):
    def __init__(self, input_size, d_model=128, nhead=4, layers=2, dropout=0.1):
        super().__init__()
        self.proj = nn.Linear(input_size, d_model)
        enc = nn.TransformerEncoderLayer(d_model, nhead, dim_feedforward=256,
                                         dropout=dropout, batch_first=True)
        self.enc  = nn.TransformerEncoder(enc, layers)
        self.fc   = nn.Linear(d_model, 1)
    def forward(self, x):
        return self.fc(self.enc(self.proj(x))[:, -1, :]).squeeze(-1)

# ── 공통 유틸 ────────────────────────────────────────────────────────────
def batch_predict(model, X, device, bs=256):
    model.eval()
    preds = []
    with torch.no_grad():
        for (xb,) in DataLoader(TensorDataset(torch.FloatTensor(X)), batch_size=bs):
            preds.append((torch.sigmoid(model(xb.to(device))) > 0.5).cpu().numpy().astype(int))
    return np.concatenate(preds)

def train_nn(ModelClass, X_train, X_test, y_train, name,
             epochs=30, bs=256, lr=1e-3, patience=5):
    # 마지막 10%를 validation으로 사용 (시계열 순서 유지)
    n_val      = max(1, int(len(X_train) * 0.1))
    X_tr, X_val = X_train[:-n_val], X_train[-n_val:]
    y_tr, y_val = y_train[:-n_val], y_train[-n_val:]

    loader     = DataLoader(TensorDataset(torch.FloatTensor(X_tr),  torch.FloatTensor(y_tr)),
                            batch_size=bs, shuffle=True)
    val_loader = DataLoader(TensorDataset(torch.FloatTensor(X_val), torch.FloatTensor(y_val)),
                            batch_size=bs * 2, shuffle=False)

    model = ModelClass(X_train.shape[2]).to(DEVICE)

    # 클래스 불균형 보정: Down 비율 / Up 비율
    pos_w     = torch.tensor([(y_tr == 0).sum() / max((y_tr == 1).sum(), 1)],
                              dtype=torch.float32).to(DEVICE)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_w)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    best_val_loss = float('inf')
    best_state    = None
    no_improve    = 0

    for epoch in range(epochs):
        model.train()
        train_loss = 0
        for xb, yb in loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
        train_loss /= len(loader)

        model.eval()
        val_loss = 0
        with torch.no_grad():
            for xb, yb in val_loader:
                val_loss += criterion(model(xb.to(DEVICE)), yb.to(DEVICE)).item()
        val_loss /= len(val_loader)

        print(f'  {name} Epoch {epoch+1:2d}/{epochs}  '
              f'train={train_loss:.4f}  val={val_loss:.4f}')

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state    = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            no_improve    = 0
        else:
            no_improve += 1
            if no_improve >= patience:
                print(f'  → Early stopping at epoch {epoch+1}  (best val={best_val_loss:.4f})')
                break

    model.load_state_dict(best_state)
    torch.save(model.state_dict(), os.path.join(MODEL_DIR, f'{name.lower()}.pt'))
    return batch_predict(model, X_test, DEVICE, bs)

print('모델 정의 완료')
print('  LSTM hidden=128 / Transformer d_model=128,ff=256')
print('  Early stopping patience=5 / pos_weight 클래스 균형 보정 활성화')

In [ ]:
# ── ARIMA (AR(5) baseline) ──────────────────────────────────────────────
def train_arima(X_train, X_test, y_train, y_test):
    from statsmodels.tsa.ar_model import AutoReg
    p            = 5
    lr_idx       = 0  # log_return은 FEATURE_COLS[0]
    train_series = X_train[:, -1, lr_idx]
    neutral      = float(np.mean(train_series))
    print(f'  train_series: {len(train_series):,}개  neutral threshold: {neutral:.4f}')
    try:
        print('  AR(5) 계수 추정 중...')
        fitted    = AutoReg(train_series, lags=p, old_names=False).fit()
        intercept = fitted.params[0]
        ar_coefs  = fitted.params[1:]
        print(f'  intercept={intercept:.4f}  AR coefs={ar_coefs.round(4).tolist()}')

        windows   = X_test[:, -p:, lr_idx]
        forecasts = intercept + windows @ ar_coefs[::-1]
        preds     = (forecasts > neutral).astype(int)
        up_ratio  = preds.mean()
        print(f'  예측 완료 — 상승(1): {preds.sum():,} ({up_ratio:.1%})  '
              f'하락(0): {(preds==0).sum():,} ({1-up_ratio:.1%})')
    except Exception as e:
        print(f'  ARIMA 실패 (zeros로 대체): {e}')
        preds = np.zeros(len(y_test), dtype=int)
    return preds

# ── LSTM ────────────────────────────────────────────────────────────────
class LSTMClassifier(nn.Module):
    def __init__(self, input_size, hidden=64, layers=2, dropout=0.2):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden, layers,
                            batch_first=True, dropout=dropout)
        self.fc   = nn.Linear(hidden, 1)
    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :]).squeeze(-1)

# ── Transformer ─────────────────────────────────────────────────────────
class TransformerClassifier(nn.Module):
    def __init__(self, input_size, d_model=64, nhead=4, layers=2, dropout=0.1):
        super().__init__()
        self.proj = nn.Linear(input_size, d_model)
        enc = nn.TransformerEncoderLayer(d_model, nhead, dim_feedforward=128,
                                         dropout=dropout, batch_first=True)
        self.enc  = nn.TransformerEncoder(enc, layers)
        self.fc   = nn.Linear(d_model, 1)
    def forward(self, x):
        return self.fc(self.enc(self.proj(x))[:, -1, :]).squeeze(-1)

# ── 공통 유틸 ────────────────────────────────────────────────────────────
def batch_predict(model, X, device, bs=256):
    model.eval()
    preds = []
    with torch.no_grad():
        for (xb,) in DataLoader(TensorDataset(torch.FloatTensor(X)), batch_size=bs):
            preds.append((torch.sigmoid(model(xb.to(device))) > 0.5).cpu().numpy().astype(int))
    return np.concatenate(preds)

def train_nn(ModelClass, X_train, X_test, y_train, name, epochs=20, bs=256, lr=1e-3):
    loader = DataLoader(
        TensorDataset(torch.FloatTensor(X_train), torch.FloatTensor(y_train)),
        batch_size=bs, shuffle=True
    )
    model     = ModelClass(X_train.shape[2]).to(DEVICE)
    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    model.train()
    for epoch in range(epochs):
        total = 0
        for xb, yb in loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            optimizer.step()
            total += loss.item()
        print(f'  {name} Epoch {epoch+1}/{epochs}  Loss: {total/len(loader):.4f}')
    torch.save(model.state_dict(), os.path.join(MODEL_DIR, f'{name.lower()}.pt'))
    return batch_predict(model, X_test, DEVICE, bs)

print('모델 정의 완료')

In [ ]:
PRED_PATH = os.path.join(DATA_DIR, 'predictions.npz')
results   = {}

# Step 4를 스킵했거나 런타임이 재시작된 경우 dataset.npz에서 자동 로드
try:
    X_train
except NameError:
    _d = np.load(os.path.join(DATA_DIR, 'dataset.npz'))
    X_train, X_test = _d['X_train'], _d['X_test']
    y_train, y_test = _d['y_train'], _d['y_test']
    print(f'dataset.npz 로드: Train={X_train.shape}, Test={X_test.shape}')

print('[1/3] ARIMA 학습...')
results['arima'] = train_arima(X_train, X_test, y_train, y_test)

print('\n[2/3] LSTM 학습...')
results['lstm'] = train_nn(LSTMClassifier, X_train, X_test, y_train, 'LSTM')

print('\n[3/3] Transformer 학습...')
results['transformer'] = train_nn(TransformerClassifier, X_train, X_test, y_train, 'Transformer')

np.savez(PRED_PATH, y_test=y_test, **results)
print(f'\n예측 저장: {PRED_PATH}')

from sklearn.metrics import accuracy_score, f1_score, classification_report

pred_data = np.load(PRED_PATH)

# log_returns_raw 로드 (없으면 금융 지표 skip)
_ds = np.load(DATASET_PATH)
log_returns_raw = _ds['log_returns_raw'] if 'log_returns_raw' in _ds.files else None

eval_results = []

for name in ['arima', 'lstm', 'transformer']:
    if name not in pred_data:
        continue
    yp  = pred_data[name]
    acc = accuracy_score(y_test, yp)
    f1  = f1_score(y_test, yp, average='weighted', zero_division=0)

    print(f'\n{"="*45}')
    print(f'  {name.upper()}')
    print(f'  Accuracy: {acc:.4f}  |  F1: {f1:.4f}')

    row = {'model': name.upper(), 'accuracy': acc, 'f1': f1}

    if log_returns_raw is not None:
        sr     = log_returns_raw * yp           # 매수 시에만 수익 반영
        cum    = float(np.exp(sr.sum()) - 1)    # 누적 로그수익률 → 단순 수익률
        sharpe = float(sr.mean() / sr.std() * np.sqrt(252)) if sr.std() > 0 else 0.0
        print(f'  Cumulative Return: {cum*100:.2f}%  |  Sharpe: {sharpe:.4f}')
        row.update({'cum_return': cum, 'sharpe': sharpe})
    else:
        print('  금융 지표: log_returns_raw 없음 (dataset 재생성 또는 복원 셀 실행 필요)')

    print(classification_report(y_test, yp, target_names=['Down', 'Up']))
    eval_results.append(row)

df_eval = pd.DataFrame(eval_results)
print('\n모델 비교')
print(df_eval.to_string(index=False))

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, classification_report

pred_data   = np.load(PRED_PATH)
log_returns = X_test[:, -1, 0]  # 마지막 타임스텝의 log_return (금융 지표용)

eval_results = []

for name in ['arima', 'lstm', 'transformer']:
    if name not in pred_data:
        continue
    yp  = pred_data[name]
    acc = accuracy_score(y_test, yp)
    f1  = f1_score(y_test, yp, average='weighted', zero_division=0)
    sr  = log_returns * yp
    cum = float(np.exp(sr.cumsum())[-1] - 1)
    sharpe = float(sr.mean() / sr.std() * np.sqrt(252)) if sr.std() != 0 else 0.0

    print(f'\n{"="*45}')
    print(f'  {name.upper()}')
    print(f'  Accuracy: {acc:.4f}  |  F1: {f1:.4f}')
    print(f'  Cumulative Return: {cum*100:.2f}%  |  Sharpe: {sharpe:.4f}')
    print(classification_report(y_test, yp, target_names=['Down', 'Up']))
    eval_results.append({'model': name.upper(), 'accuracy': acc, 'f1': f1,
                         'cum_return': cum, 'sharpe': sharpe})

df_eval = pd.DataFrame(eval_results)
print('\n모델 비교')
print(df_eval.to_string(index=False))

## Step 7: 시각화

## 실험: Sentiment Feature Ablation

감성 피처(인덱스 12~17)를 제거했을 때 성능이 어떻게 변하는지 비교합니다.  
뉴스 감성 분석이 실제로 예측에 기여하는지 검증합니다.

In [ ]:
# FEATURE_COLS 기준 감성 피처 인덱스: 12~17
# ['sentiment_mean_weighted', 'sentiment_std_weighted',
#  'sentiment_lag1', 'sentiment_lag2', 'sentiment_change', 'news_count_zscore_20']
SENT_IDXS = list(range(12, 18))

X_train_ns = X_train.copy(); X_train_ns[:, :, SENT_IDXS] = 0
X_test_ns  = X_test.copy();  X_test_ns[:, :, SENT_IDXS]  = 0

print('=== Sentiment Ablation: 감성 피처 제거 후 재학습 ===\n')
print('[1/2] LSTM (감성 없음)...')
preds_lstm_ns = train_nn(LSTMClassifier, X_train_ns, X_test_ns, y_train, 'LSTM_NoSent')

print('\n[2/2] Transformer (감성 없음)...')
preds_tf_ns = train_nn(TransformerClassifier, X_train_ns, X_test_ns, y_train, 'Transformer_NoSent')

# 비교 테이블
print('\n=== 감성 피처 유무 성능 비교 ===')
abl_rows = []
for model_name, p_with, p_without in [
    ('LSTM',        results['lstm'],        preds_lstm_ns),
    ('Transformer', results['transformer'], preds_tf_ns),
]:
    for tag, p in [('w/ sentiment', p_with), ('w/o sentiment', p_without)]:
        abl_rows.append({
            'model':    f'{model_name} ({tag})',
            'accuracy': accuracy_score(y_test, p),
            'f1':       f1_score(y_test, p, average='weighted', zero_division=0),
            'up_recall': float((p[y_test == 1] == 1).mean()),
        })

df_abl = pd.DataFrame(abl_rows)
print(df_abl.to_string(index=False))

# 시각화
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, metric in zip(axes, ['accuracy', 'f1']):
    bars = ax.bar(range(len(df_abl)), df_abl[metric], color=['#4C72B0','#DD8452']*2)
    ax.set_xticks(range(len(df_abl)))
    ax.set_xticklabels(df_abl['model'], rotation=15, ha='right')
    ax.set_ylim(0.4, 0.7)
    ax.set_title(metric.upper())
    ax.set_ylabel('Score')
    for bar, val in zip(bars, df_abl[metric]):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                f'{val:.3f}', ha='center', va='bottom', fontsize=9)
plt.suptitle('Sentiment Feature Ablation')
plt.tight_layout()
plt.savefig(os.path.join(RESULT_DIR, 'ablation_sentiment.png'), dpi=150)
plt.show()

In [ ]:
from sklearn.metrics import (
    precision_score, recall_score, confusion_matrix, ConfusionMatrixDisplay
)

model_names = [n for n in ['arima', 'lstm', 'transformer'] if n in pred_data.files]

# 성능 지표 막대그래프
rows = []
for name in model_names:
    yp = pred_data[name]
    rows.append({
        'model':     name.upper(),
        'accuracy':  accuracy_score(y_test, yp),
        'precision': precision_score(y_test, yp, average='weighted', zero_division=0),
        'recall':    recall_score(y_test, yp, average='weighted', zero_division=0),
        'f1_score':  f1_score(y_test, yp, average='weighted', zero_division=0),
    })
df_metrics = pd.DataFrame(rows)

metric_cols = ['accuracy', 'precision', 'recall', 'f1_score']
x     = np.arange(len(df_metrics))
width = 0.18
plt.figure(figsize=(10, 6))
for i, col in enumerate(metric_cols):
    plt.bar(x + (i - 1.5) * width, df_metrics[col], width, label=col)
plt.xticks(x, df_metrics['model'])
plt.ylim(0, 1)
plt.ylabel('Score')
plt.title('Model Classification Performance')
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(RESULT_DIR, 'metrics.png'), dpi=150)
plt.show()

# Confusion Matrix
fig, axes = plt.subplots(1, len(model_names), figsize=(5 * len(model_names), 5))
if len(model_names) == 1:
    axes = [axes]
for ax, name in zip(axes, model_names):
    cm = confusion_matrix(y_test, pred_data[name])
    ConfusionMatrixDisplay(cm, display_labels=['Down', 'Up']).plot(
        ax=ax, colorbar=False, values_format='d'
    )
    ax.set_title(name.upper())
plt.tight_layout()
plt.savefig(os.path.join(RESULT_DIR, 'confusion_matrices.png'), dpi=150)
plt.show()

# 예측 분포 (상승 vs 하락 비율)
dist_rows = []
for name in model_names:
    yp = pred_data[name]
    dist_rows.append({'model': name.upper(), 'Down': (yp == 0).mean(), 'Up': (yp == 1).mean()})
df_dist = pd.DataFrame(dist_rows)

x2 = np.arange(len(df_dist))
plt.figure(figsize=(8, 5))
plt.bar(x2 - 0.175, df_dist['Down'], 0.35, label='Down')
plt.bar(x2 + 0.175, df_dist['Up'],   0.35, label='Up')
plt.xticks(x2, df_dist['model'])
plt.ylim(0, 1)
plt.ylabel('Prediction Ratio')
plt.title('Prediction Distribution by Model')
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(RESULT_DIR, 'distribution.png'), dpi=150)
plt.show()

# 결과 CSV 저장
df_metrics.to_csv(os.path.join(RESULT_DIR, 'metrics.csv'), index=False, encoding='utf-8-sig')
print('시각화 완료 — 결과는 RESULT_DIR 참고')